In [172]:
import numpy as np
import pandas as pd
np.random.seed(100)

## Fake Longitudinal Data

Here I'm going to make some fake longitudinal patient data. The idea is to have irregular doctor visits with several features recorded at each visit. Eventually, I want to represent each patient as a graph and predict whether that patient was ever sick.

We will have \(N\) subjects, with

$$
N=500.
$$

In [173]:
N = 500

N

500

Each patient will have some number of visits, \(n_i\). We will randomly choose between 3 and 10 visits for each patient:

$$
n_i \sim U\{3,\ldots,10\}.
$$

In [174]:
n_i = np.random.randint(3, 11, size=N)

n_i[:10]

array([ 3,  3,  6, 10, 10, 10,  3,  5,  9,  7], dtype=int32)

At each visit, we will eventually record \(p=5\) features:

$$
p=5.
$$

In [175]:
p = 5

p

5

Each patient will have two underlying variables.

The first, \(z_i\), represents the number of times that patient gets sick:

$$
z_i \sim \text{Poisson}(1).
$$

The second, \(v_i\), represents how intense their sickness tends to be:

$$
v_i \sim N(0,1).
$$

Larger values of \(v_i\) will correspond to more intense symptoms.

In [176]:
z = np.random.poisson(lam=1, size=N)

z[:10]

array([1, 3, 2, 1, 1, 3, 1, 3, 2, 0], dtype=int32)

In [177]:
v = np.random.normal(0, 1, size=N)

v[:10]

array([-2.14185161, -0.68365014, -1.18079803,  1.18100216, -1.06605526,
       -0.74304592, -0.88592525, -0.49581834,  0.52738769, -0.30175139])

In [178]:
patient_data = pd.DataFrame({
    "subject": np.arange(1, N + 1),
    "n_visits": n_i,
    "z": z,
    "v": v
})

patient_data.head()

,subject,n_visits,z,v
0,1,3,1,-2.141852
1,2,3,3,-0.683650
2,3,6,2,-1.180798
3,4,10,1,1.181002
4,5,10,1,-1.066055


The number of sick visits cannot be greater than the total number of visits.

Therefore, define

$$
s_i = \min(z_i,n_i),
$$

where \(s_i\) is the actual number of sick visits that we will assign to patient \(i\).

In [179]:
s_i = np.minimum(z, n_i)

patient_data["n_sick"] = s_i

patient_data.head()

,subject,n_visits,z,v,n_sick
0,1,3,1,-2.141852,1
1,2,3,3,-0.683650,3
2,3,6,2,-1.180798,2
3,4,10,1,1.181002,1
4,5,10,1,-1.066055,1


Now I will create the long dataframe.

There will be one row for each visit of each patient. For now, I only need the subject number and visit number.

In [180]:
rows = []

for i in range(N):

    for j in range(n_i[i]):

        rows.append({
            "subject": i + 1,
            "visit": j + 1
        })

data = pd.DataFrame(rows)

data.head()

,subject,visit
0,1,1
1,1,2
2,1,3
3,2,1
4,2,2


For each patient, \(s_i\) of their visits will randomly be designated as sick visits.

Define

$$
S_{ij}
=
\begin{cases}
1, & \text{if visit }j\text{ is a sick visit},\\
0, & \text{otherwise}.
\end{cases}
$$

The sick visits will be selected uniformly from all visits for that patient.

In [181]:
data["sick_visit"] = 0

for i in range(N):

    subject = i + 1

    # Find all rows belonging to this patient
    patient_rows = data.index[data["subject"] == subject].to_numpy()

    # Number of sick visits for this patient
    n_sick = s_i[i]

    if n_sick > 0:

        sick_rows = np.random.choice(
            patient_rows,
            size=n_sick,
            replace=False
        )

        data.loc[sick_rows, "sick_visit"] = 1

data.head()

,subject,visit,sick_visit
0,1,1,1
1,1,2,0
2,1,3,0
3,2,1,1
4,2,2,1


I don't want \(v_i\) itself to determine the size of the sickness effect because \(v_i\) can be any real number.

Instead, I transform it using

$$
\sigma(v_i)
=
\frac{\exp(v_i)}
{1+\exp(v_i)}.
$$

This gives a sickness intensity between 0 and 1.

In [182]:
severity = np.exp(v) / (1 + np.exp(v))

patient_data["severity"] = severity

patient_data.head()

,subject,n_visits,z,v,n_sick,severity
0,1,3,1,-2.141852,1,0.105095
1,2,3,3,-0.683650,3,0.335447
2,3,6,2,-1.180798,2,0.234909
3,4,10,1,1.181002,1,0.765128
4,5,10,1,-1.066055,1,0.256154


The visits will occur at irregular times.

I will work with the gap between visits,

$$
\Delta t_{ij}
=
t_{ij}-t_{i,j-1}.
$$

For a regular visit, the average gap will be 2 time units.

If the visit is a sick visit, the gap will tend to be shorter:

$$
\Delta t_{ij}
\sim
\text{Exponential}
\left(
2-1.5\sigma(v_i)S_{ij}
\right).
$$

Here the parameter is the mean of the exponential distribution.

The first visit occurs at

$$
t_{i1}=0.
$$

In [183]:
data["gap"] = 0.0

for i in range(N):

    subject = i + 1

    patient_rows = data.index[data["subject"] == subject].to_numpy()

    for j in range(1, len(patient_rows)):

        row = patient_rows[j]

        sick = data.loc[row, "sick_visit"]

        mean_gap = 2 - 1.5 * severity[i] * sick

        data.loc[row, "gap"] = np.random.exponential(
            scale=mean_gap
        )

The actual visit time is the cumulative sum of the gaps:

$$
t_{ij}
=
\sum_{k=1}^{j}\Delta t_{ik}.
$$

In [184]:
data["time"] = data.groupby("subject")["gap"].cumsum()

data.head()

,subject,visit,sick_visit,gap,time
0,1,1,1,0.000000,0.000000
1,1,2,0,3.377272,3.377272
2,1,3,0,0.086067,3.463339
3,2,1,1,0.000000,0.000000
4,2,2,1,0.307695,0.307695


At each visit we observe \(p=5\) features.

Each feature has a normal value of

$$
\mu_k=5,
$$

so

$$
\boldsymbol{\mu}=(5,5,5,5,5).
$$

Every patient starts with all five features close to their normal values.

In [185]:
mu = np.array([5, 5, 5, 5, 5])

mu

array([5, 5, 5, 5, 5])

Between visits, each feature changes randomly but is pulled back toward its normal value.

We define

$$
\epsilon_{ijk}
\sim
N\left(
\alpha(\mu_k-x_{i(j-1)k}),
\sigma_\epsilon^2
\right).
$$

I will use

$$
\alpha=0.3
$$

and

$$
\sigma_\epsilon=1.
$$

In [186]:
alpha = 0.5
sigma_e = 0.5

for k in range(1, p + 1):
    data[f"x{k}"] = np.nan

data.head()

,subject,visit,sick_visit,gap,time,x1,x2,x3,x4,x5
0,1,1,1,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN
1,1,2,0,3.377272,3.377272,NaN,NaN,NaN,NaN,NaN
2,1,3,0,0.086067,3.463339,NaN,NaN,NaN,NaN,NaN
3,2,1,1,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN
4,2,2,1,0.307695,0.307695,NaN,NaN,NaN,NaN,NaN


For every patient, the first visit starts near the normal value:

$$
\mathbf{x}_{i1}
=
(5,5,5,5,5) + \delta
$$
where $\delta\sim N(0,0.5\mathbf{I}_5)$

In [187]:
for i in range(N):

    subject = i + 1

    first_row = data.index[data["subject"] == subject][0]

    data.loc[first_row, ["x1", "x2", "x3", "x4", "x5"]] = mu + np.random.normal(0, 0.5, size=p)

data.head()

,subject,visit,sick_visit,gap,time,x1,x2,x3,x4,x5
0,1,1,1,0.000000,0.000000,4.831624,4.496584,3.714200,5.993952,5.703072
1,1,2,0,3.377272,3.377272,NaN,NaN,NaN,NaN,NaN
2,1,3,0,0.086067,3.463339,NaN,NaN,NaN,NaN,NaN
3,2,1,1,0.000000,0.000000,6.236979,5.231933,5.350978,5.017351,5.747703
4,2,2,1,0.307695,0.307695,NaN,NaN,NaN,NaN,NaN


Now I update the features one visit at a time.

The first feature is a dud feature:

$$
x_{ij1}
=
x_{i(j-1)1}
+
\epsilon_{ij1}.
$$

The remaining features are affected by sickness:

$$
x_{ij2}
=
x_{i(j-1)2}
+
3\sigma(v_i)S_{ij}
+
\epsilon_{ij2},
$$

$$
x_{ij3}
=
x_{i(j-1)3}
-
2\sigma(v_i)S_{ij}
+
\epsilon_{ij3},
$$

$$
x_{ij4}
=
x_{i(j-1)4}
+
1.5\sigma(v_i)S_{ij}
+
\epsilon_{ij4},
$$

and

$$
x_{ij5}
=
x_{i(j-1)5}
+
0.75\sigma(v_i)S_{ij}
+
\epsilon_{ij5}.
$$

In [188]:
for i in range(N):

    subject = i + 1

    patient_rows = data.index[data["subject"] == subject].to_numpy()

    for j in range(1, len(patient_rows)):

        previous_row = patient_rows[j - 1]
        current_row = patient_rows[j]

        # Previous feature values
        previous_x = data.loc[
            previous_row,
            ["x1", "x2", "x3", "x4", "x5"]
        ].to_numpy(dtype=float)

        # Random mean-reverting changes
        epsilon = np.random.normal(
            loc=alpha * (mu - previous_x),
            scale=sigma_e,
            size=p
        )

        # Is this a sick visit?
        sick = data.loc[current_row, "sick_visit"]

        # Start with previous values plus random changes
        current_x = previous_x + epsilon

        # Add sickness effects
        current_x[1] += 3.0 * severity[i] * sick
        current_x[2] -= 2.0 * severity[i] * sick
        current_x[3] += 1.5 * severity[i] * sick
        current_x[4] += 0.75 * severity[i] * sick

        # Save the new values
        data.loc[
            current_row,
            ["x1", "x2", "x3", "x4", "x5"]
        ] = current_x

data.head(20)

,subject,visit,sick_visit,gap,time,x1,x2,x3,x4,x5
0,1,1,1,0.000000,0.000000,4.831624,4.496584,3.714200,5.993952,5.703072
1,1,2,0,3.377272,3.377272,5.294986,5.495530,3.938673,4.995435,4.947068
2,1,3,0,0.086067,3.463339,5.050891,4.670009,4.661214,4.792993,5.522478
3,2,1,1,0.000000,0.000000,6.236979,5.231933,5.350978,5.017351,5.747703
4,2,2,1,0.307695,0.307695,6.607205,6.219897,4.269564,5.430589,5.381619
5,2,3,1,1.616750,1.924445,5.804753,6.201448,4.634858,5.440366,6.017674
6,3,1,1,0.000000,0.000000,4.627639,5.272026,5.069991,5.300591,5.517961
7,3,2,0,0.795594,0.795594,4.484027,4.901754,4.814674,5.131084,5.032769
8,3,3,0,2.538774,3.334367,4.528657,5.262324,4.774163,4.455452,4.234460
9,3,4,0,0.948829,4.283197,3.681329,5.426173,5.168178,4.429389,4.447108


Finally, each patient gets one binary outcome indicating whether they were ever sick.

Define

$$
Y_i
=
I(z_i > 0).
$$

So

$$
Y_i =
\begin{cases}
1, & \text{if the patient was sick at least once},\\
0, & \text{if the patient was never sick}.
\end{cases}
$$

Since

$$
z_i \sim \text{Poisson}(1),
$$

we expect

$$
P(Y_i=0)
=
P(z_i=0)
=
e^{-1}
\approx 0.368.
$$

In [189]:
y = (z > 0).astype(int)

patient_data["y"] = y

patient_data.head(20)

,subject,n_visits,z,v,n_sick,severity,y
0,1,3,1,-2.141852,1,0.105095,1
1,2,3,3,-0.683650,3,0.335447,1
2,3,6,2,-1.180798,2,0.234909,1
3,4,10,1,1.181002,1,0.765128,1
4,5,10,1,-1.066055,1,0.256154,1
5,6,10,3,-0.743046,3,0.322338,1
6,7,3,1,-0.885925,1,0.291951,1
7,8,5,3,-0.495818,3,0.378524,1
8,9,9,2,0.527388,2,0.628874,1
9,10,7,0,-0.301751,0,0.425129,0


Because the outcome is defined at the patient level, I only want to count each patient once when checking the class distribution.

In [190]:
patient_data["y"].value_counts()

y
1    322
0    178
Name: count, dtype: int64

In [191]:

patient_data["y"].value_counts(normalize=True)

y
1    0.644
0    0.356
Name: proportion, dtype: float64

Also I want to check the sanity of the visits to see if I didn't mess that up.

In [192]:
data[data["visit"] > 1].groupby("sick_visit")["gap"].describe()

,count,mean,std,min,25%,50%,75%,max
sick_visit,,,,,,,,
0,2349.0,1.929624,1.999839,0.002492,0.523524,1.305147,2.652987,15.947417
1,404.0,1.214865,1.400748,0.000636,0.298712,0.753362,1.564810,8.234983


In [193]:
data[data["visit"] > 1].groupby("sick_visit")[["x1", "x2", "x3", "x4", "x5"]].mean()

,x1,x2,x3,x4,x5
sick_visit,,,,,
0,4.998941,5.116869,4.937475,5.075662,5.034224
1,5.001759,6.730109,3.842150,5.834292,5.397241


Looks alright. Let's save the data then.

In [194]:
data.to_csv("fake_longitudinal_data.csv", index=False)
patient_data.to_csv("fake_patient_data.csv", index=False)